# 10 · Statistical analysis per research question (pre-registered plan)
Effect size everywhere = **ΔAUROC** with a 95% paired, class-stratified, document-level bootstrap CI (seed-averaged AUROC inside each resample). p-values: bootstrap (two-sided) and, for single-score comparisons, DeLong. **Holm** correction within each RQ family. Cliff's δ (= 2·AUROC − 1) is reported for the main detectors.

| RQ | Comparison | Unit of analysis |
|---|---|---|
| RQ1 | E2/all vs E1/in-language, on each Turkic target | test docs of the target |
| RQ2 | 6×6 transfer matrix (E1); two-way fixed-effects model logit(AUROC) ~ same_script + same_family + source + target; doc-bootstrap CIs + exact permutation test over language labels; LOLO cost (E2/no-L vs E2/all) | language pairs |
| RQ3 | E3 (family held out) vs E2/all, on the held-out family's test texts; external COLING-kk (GPT-4o) with OpenAI held out | test docs |
| RQ4 | attacked vs clean (same original docs); zero-shot vs fine-tuned difference-in-differences; defence recovery | original test docs |
| RQ5 | Kazakh-adapted vs generic scorer for each zero-shot method | test docs per language |

**Outputs:** `09_statistics/rq*.csv`.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "10_statistical_analysis"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
import itertools, statsmodels.formula.api as smf
B = CFG["stats"]["bootstrap_B"]; ALPHA = CFG["stats"]["alpha"]
EVAL = load_eval_texts()
reg = load_registry(); reg = reg[reg.status == "done"]
_cache = {}
def preds(run_id):
    if run_id not in _cache:
        _cache[run_id] = pd.read_parquet(pred_path(run_id)).set_index("doc_id").score
    return _cache[run_id]

def runs(exp, detector, train, heldout="none"):
    r = reg[(reg.exp == exp) & (reg.detector == detector) & (reg.train == train) & (reg.heldout == heldout)]
    return list(r.run_id)

def aligned(eval_h, eval_m, lang=None, family=None):
    """Docs of an evaluation set aligned by orig_doc_id: returns frame with columns orig_doc_id, label, doc_id."""
    h = EVAL[(EVAL.eval_set == eval_h) & (EVAL.label == 0)]; m = EVAL[(EVAL.eval_set == eval_m) & (EVAL.label == 1)]
    d = pd.concat([h, m])
    if lang:
        d = d[d.lang == lang]
    if family:
        d = d[(d.label == 0) | (d.family == family)]
    return d.sort_values("orig_doc_id").reset_index(drop=True)

def S_of(run_ids, docs):
    M = np.vstack([preds(r).reindex(docs.doc_id).values for r in run_ids])
    assert not np.isnan(M).any(), "missing scores — was this evaluation set scored by these runs?"
    return M

def boot_aucs(y, mats, seed=0):
    """Seed-averaged AUROC for each score matrix on the same B resamples -> array (len(mats), B) and point estimates."""
    idx = strat_boot_idx(y, B, seed)
    boots = np.array([np.mean([auroc_boot(y, s, idx) for s in M], axis=0) for M in mats])
    pts = np.array([np.mean([auroc(y, s) for s in M]) for M in mats])
    return pts, boots

def contrast(pts, boots, w):
    w = np.asarray(w, float); d = boots.T @ w; pt = float(pts @ w)
    lo, hi = np.quantile(d, [ALPHA / 2, 1 - ALPHA / 2]); p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return pt, float(lo), float(hi), float(max(min(1.0, p), 1 / B))

FT_DETS = [d for d in ["mdeberta", "xlmr", "ngram", "stylo"] if (reg.detector == d).any()]
TURKIC = CFG["turkic_targets"]

## RQ1 — multilingual vs in-language training (Turkic targets)

In [ ]:
out = []
for det, L in itertools.product(FT_DETS, TURKIC):
    a, b = runs("E2", det, "all"), runs("E1", det, L)
    if not a or not b:
        continue
    d = aligned("test", "test", L); y = d.label.values
    pts, boots = boot_aucs(y, [S_of(a, d), S_of(b, d)])
    dl, lo, hi, p = contrast(pts, boots, [1, -1])
    _, _, z, p_dl = delong_test(y, S_of(a, d).mean(0), S_of(b, d).mean(0))
    out.append({"detector": det, "target": L, "auroc_multi": pts[0], "auroc_mono": pts[1], "delta": dl, "ci_lo": lo, "ci_hi": hi,
                "p_boot": p, "p_delong_seedavg": p_dl, "cliffs_delta_multi": cliffs_delta_from_auc(pts[0])})
rq1 = pd.DataFrame(out)
if len(rq1):
    rq1["p_holm"] = holm(rq1.p_boot)
rq1.to_csv(P["stats"] / "rq1_multi_vs_mono.csv", index=False); rq1.round(4)

## RQ2 — cross-lingual transfer: family vs script
Transfer matrix from monolingual (E1) models; regression on the 30 off-diagonal pairs per detector.

In [ ]:
fam = {l: LANG_META[l]["family"] for l in LANGS}; scr = {l: LANG_META[l]["script"] for l in LANGS}
mat_rows, reg_rows, boot_rows = [], [], []

def fit_fe(df):
    df = df.assign(logit=np.log(df.auc.clip(1e-4, 1 - 1e-4) / (1 - df.auc.clip(1e-4, 1 - 1e-4))))
    m = smf.ols("logit ~ same_script + same_family + C(src) + C(tgt)", data=df).fit()
    return m.params["same_script"], m.params["same_family"]

for det in FT_DETS:
    pair_pts, pair_boot = {}, {}
    for tgt in LANGS:
        d = aligned("test", "test", tgt); y = d.label.values
        srcs = [s for s in LANGS if runs("E1", det, s)]
        pts, boots = boot_aucs(y, [S_of(runs("E1", det, s), d) for s in srcs], seed=LANGS.index(tgt))
        for s, pt, bt in zip(srcs, pts, boots):
            pair_pts[(s, tgt)] = pt; pair_boot[(s, tgt)] = bt
            mat_rows.append({"detector": det, "src": s, "tgt": tgt, "auroc": pt,
                             "ci_lo": float(np.quantile(bt, ALPHA / 2)), "ci_hi": float(np.quantile(bt, 1 - ALPHA / 2))})
    off = [(s, t) for (s, t) in pair_pts if s != t]
    if len(off) < 12:
        continue
    base = pd.DataFrame([{"src": s, "tgt": t, "same_script": int(scr[s] == scr[t]), "same_family": int(fam[s] == fam[t])} for s, t in off])
    b_script, b_family = fit_fe(base.assign(auc=[pair_pts[k] for k in off]))
    # (a) document-bootstrap CI for the coefficients
    bs = np.array([fit_fe(base.assign(auc=[pair_boot[k][i] for k in off])) for i in range(B)])
    # (b) exact permutation test over which languages carry each label (keeps the 2x2 structure size)
    def perm_p(coef_idx, labels, obs):
        stats = []
        for combo in itertools.combinations(LANGS, sum(v == list(labels.values())[0] for v in labels.values())):
            lab = {l: ("A" if l in combo else "B") for l in LANGS}
            if coef_idx == 0:
                df = base.assign(same_script=[int(lab[s] == lab[t]) for s, t in off])
            else:
                df = base.assign(same_family=[int(lab[s] == lab[t]) for s, t in off])
            stats.append(fit_fe(df.assign(auc=[pair_pts[k] for k in off]))[coef_idx])
        stats = np.abs(np.array(stats)); return float((stats >= abs(obs) - 1e-12).mean()), len(stats)
    p_scr, n_scr = perm_p(0, {l: scr[l] for l in LANGS}, b_script)
    p_fam, n_fam = perm_p(1, fam, b_family)
    qs = np.quantile(bs[:, 0], [ALPHA / 2, 1 - ALPHA / 2]); qf = np.quantile(bs[:, 1], [ALPHA / 2, 1 - ALPHA / 2])
    reg_rows.append({"detector": det, "coef_same_script": b_script, "ci_lo_script": float(qs[0]), "ci_hi_script": float(qs[1]),
                     "perm_p_script": p_scr, "n_perm_script": n_scr,
                     "coef_same_family": b_family, "ci_lo_family": float(qf[0]), "ci_hi_family": float(qf[1]),
                     "perm_p_family": p_fam, "n_perm_family": n_fam,
                     "boot_p_family_gt_script": float((bs[:, 1] <= bs[:, 0]).mean())})
pd.DataFrame(mat_rows).to_csv(P["stats"] / "rq2_transfer_matrix.csv", index=False)
rq2 = pd.DataFrame(reg_rows); rq2.to_csv(P["stats"] / "rq2_family_vs_script.csv", index=False); rq2

**Interpretation guard:** with 6 languages the exact permutation test is coarse: 15 distinct family assignments (minimum p ≈ 0.067) and only 10 distinct script assignments (a 3/3 split and its complement give the same pairing, so minimum p = 0.10). It therefore cannot reach p < 0.05 at the language level. The bootstrap CI reflects document sampling; the permutation p reflects language-level generalisation. Report both and do not over-claim.

In [ ]:
# LOLO: cost of never seeing the target language (E2/no-L) vs seeing it (E2/all)
out = []
for det, L in itertools.product(FT_DETS, TURKIC):
    a, b = runs("E2", det, f"no-{L}"), runs("E2", det, "all")
    if not a or not b:
        continue
    d = aligned("test", "test", L); y = d.label.values
    pts, boots = boot_aucs(y, [S_of(a, d), S_of(b, d)])
    dl, lo, hi, p = contrast(pts, boots, [1, -1])
    out.append({"detector": det, "target": L, "auroc_lolo": pts[0], "auroc_all": pts[1], "delta": dl, "ci_lo": lo, "ci_hi": hi, "p_boot": p})
lolo = pd.DataFrame(out)
if len(lolo):
    lolo["p_holm"] = holm(lolo.p_boot)
lolo.to_csv(P["stats"] / "rq2_lolo.csv", index=False); lolo.round(4)

## RQ3 — unseen generator families (leave-one-family-out)

In [ ]:
out = []
for det in FT_DETS:
    for f in sorted(set(reg[reg.exp == "E3"].heldout)):
        a, b = runs("E3", det, "all", f), runs("E2", det, "all")
        if not a or not b:
            continue
        for L in LANGS + [None]:
            d = aligned("test", "test", L, family=f); y = d.label.values
            pts, boots = boot_aucs(y, [S_of(a, d), S_of(b, d)])
            dl, lo, hi, p = contrast(pts, boots, [1, -1])
            out.append({"detector": det, "heldout_family": f, "lang": L or "all", "auroc_unseen": pts[0], "auroc_seen": pts[1],
                        "delta": dl, "ci_lo": lo, "ci_hi": hi, "p_boot": p})
rq3 = pd.DataFrame(out)
if len(rq3):
    m = rq3.lang != "all"; rq3.loc[m, "p_holm"] = holm(rq3[m].p_boot)
rq3.to_csv(P["stats"] / "rq3_lofo.csv", index=False)

# Turkic vs non-Turkic unseen-family penalty (difference of pooled deltas, independent groups)
out = []
for det in FT_DETS:
    for f in sorted(set(reg[reg.exp == "E3"].heldout)):
        a, b = runs("E3", det, "all", f), runs("E2", det, "all")
        if not a or not b:
            continue
        ds_ = []
        for grp in (TURKIC, [l for l in LANGS if l not in TURKIC]):
            d = aligned("test", "test", family=f); d = d[d.lang.isin(grp)]; y = d.label.values
            pts, boots = boot_aucs(y, [S_of(a, d), S_of(b, d)], seed=len(grp))
            ds_.append((pts @ [1, -1], boots.T @ [1, -1]))
        diff = ds_[0][1] - ds_[1][1]
        out.append({"detector": det, "heldout_family": f, "delta_turkic": ds_[0][0], "delta_nonturkic": ds_[1][0],
                    "did": ds_[0][0] - ds_[1][0], "ci_lo": float(np.quantile(diff, ALPHA / 2)), "ci_hi": float(np.quantile(diff, 1 - ALPHA / 2))})
pd.DataFrame(out).to_csv(P["stats"] / "rq3_turkic_vs_nonturkic.csv", index=False)

# External: COLING-kk (GPT-4o) — OpenAI-held-out models vs all-family models, raw vs normalised text
out = []
for det in FT_DETS:
    for ver in ("raw", "norm"):
        d = aligned(f"ext_coling_{ver}", f"ext_coling_{ver}"); y = d.label.values
        cands = {"E2_all": runs("E2", det, "all"), "E3_no_openai": runs("E3", det, "all", "openai"), "E1_kk": runs("E1", det, "kk")}
        cands = {k: v for k, v in cands.items() if v}
        pts, boots = boot_aucs(y, [S_of(v, d) for v in cands.values()])
        for (k, _), pt, bt in zip(cands.items(), pts, boots):
            out.append({"detector": det, "version": ver, "model": k, "auroc": pt,
                        "ci_lo": float(np.quantile(bt, ALPHA / 2)), "ci_hi": float(np.quantile(bt, 1 - ALPHA / 2))})
for r in reg[reg.exp == "E5"].itertuples():
    for ver in ("raw", "norm"):
        d = aligned(f"ext_coling_{ver}", f"ext_coling_{ver}"); y = d.label.values
        a, lo, hi = boot_ci_auroc(y, S_of([r.run_id], d), B=B)
        out.append({"detector": r.detector, "version": ver, "model": "zero-shot", "auroc": a, "ci_lo": lo, "ci_hi": hi})
pd.DataFrame(out).to_csv(P["stats"] / "external_coling_kk.csv", index=False)

## RQ4 — robustness (attacked vs clean on the same original documents)

In [ ]:
ATTACKS = [(n, n) for n in ("T1_para", "T2_bt", "T3_h05", "T3_h15", "T4_t05", "T4_t15")] + \
          [(n + "both", n) for n in ("T3_h05", "T3_h15", "T4_t05", "T4_t15")]
ATTACKS = [(h, m) for h, m in ATTACKS if (EVAL.eval_set == m).any()]
main = {det: runs("E2", det, "all") for det in FT_DETS}
main |= {r.detector: [r.run_id] for r in reg[(reg.exp == "E5") & reg.detector.str.match(r"(binoculars|fastdetect)-")].itertuples()}
main = {k: v for k, v in main.items() if v}

def pair_clean_attack(h_set, m_set, lang):
    """Aligned arrays: clean test docs and attacked docs with identical orig_doc_id order."""
    c = aligned("test", "test", lang); a = aligned("test" if h_set == m_set else h_set, m_set, lang)
    common = sorted(set(c.orig_doc_id) & set(a.orig_doc_id))
    c = c.set_index("orig_doc_id").loc[common].reset_index(); a = a.set_index("orig_doc_id").loc[common].reset_index()
    return c, a

out = []
for (h_set, m_set), L in itertools.product(ATTACKS, LANGS + [None]):
    c, a = pair_clean_attack(h_set, m_set, L); y = c.label.values
    if len(c) == 0:
        continue
    mats = [];  names = list(main)
    for det in names:
        mats += [S_of(main[det], c), S_of(main[det], a)]
    pts, boots = boot_aucs(y, mats, seed=7)
    for i, det in enumerate(names):
        w = np.zeros(len(mats)); w[2 * i + 1], w[2 * i] = 1, -1
        dl, lo, hi, p = contrast(pts, boots, w)
        out.append({"attack": h_set, "lang": L or "all", "detector": det, "auroc_clean": pts[2 * i], "auroc_attacked": pts[2 * i + 1],
                    "delta": dl, "ci_lo": lo, "ci_hi": hi, "p_boot": p})
    # H4: zero-shot vs fine-tuned degradation (difference-in-differences)
    for zs, ft in itertools.product([n for n in names if "-" in n], [n for n in names if n in FT_DETS]):
        i, j = names.index(zs), names.index(ft)
        w = np.zeros(len(mats)); w[2 * i + 1], w[2 * i], w[2 * j + 1], w[2 * j] = 1, -1, -1, 1
        dl, lo, hi, p = contrast(pts, boots, w)
        out.append({"attack": h_set, "lang": L or "all", "detector": f"DiD[{zs} - {ft}]", "delta": dl, "ci_lo": lo, "ci_hi": hi, "p_boot": p})
rq4 = pd.DataFrame(out)
m = (rq4.lang != "all") & ~rq4.detector.str.startswith("DiD")
rq4.loc[m, "p_holm"] = holm(rq4[m].p_boot)
rq4.to_csv(P["stats"] / "rq4_robustness.csv", index=False)

# H4b: defence recovery (T3 with vs without Unicode defence) and defence cost on clean text
out = []
for tag in ("T3_h05", "T3_h15"):
    if (tag + "_def") not in set(EVAL.eval_set):
        continue
    for det, rids in main.items():
        for L in LANGS + [None]:
            c, a = pair_clean_attack(tag, tag, L)            # clean docs / attacked machine docs (clean humans)
            dfn = aligned("test_def", tag + "_def", L).set_index("orig_doc_id").loc[a.orig_doc_id].reset_index()
            y = a.label.values
            pts, boots = boot_aucs(y, [S_of(rids, a), S_of(rids, dfn), S_of(rids, c)])
            rec, lo, hi, p = contrast(pts, boots, [-1, 1, 0])
            out.append({"attack": tag, "detector": det, "lang": L or "all", "auroc_attacked": pts[0], "auroc_defended": pts[1],
                        "auroc_clean": pts[2], "recovery": rec, "ci_lo": lo, "ci_hi": hi, "p_boot": p})
pd.DataFrame(out).to_csv(P["stats"] / "rq4_defence.csv", index=False)
rq4[(rq4.lang == "all") & ~rq4.detector.str.startswith("DiD")].pivot_table(index="detector", columns="attack", values="delta").round(3)

## RQ5 — language-adapted (Kazakh) vs generic scoring LM

In [ ]:
out = []
for meth in ("binoculars", "fastdetect", "loglik", "logrank"):
    a = reg[(reg.exp == "E5") & (reg.detector == f"{meth}-kazakh")].run_id.tolist()
    b = reg[(reg.exp == "E5") & (reg.detector == f"{meth}-generic")].run_id.tolist()
    if not a or not b:
        continue
    for L in LANGS:
        for ev in ("test", "ext_coling_norm") if L == "kk" else ("test",):
            d = aligned(ev, ev, L); y = d.label.values
            if len(d) == 0:
                continue
            pts, boots = boot_aucs(y, [S_of(a, d), S_of(b, d)])
            dl, lo, hi, p = contrast(pts, boots, [1, -1])
            _, _, z, p_dl = delong_test(y, S_of(a, d)[0], S_of(b, d)[0])
            out.append({"method": meth, "lang": L, "eval": ev, "auroc_kazakh": pts[0], "auroc_generic": pts[1], "delta": dl,
                        "ci_lo": lo, "ci_hi": hi, "p_boot": p, "p_delong": p_dl, "primary": meth in ("binoculars", "fastdetect") and ev == "test"})
rq5 = pd.DataFrame(out)
if len(rq5):
    m = rq5.primary; rq5.loc[m, "p_holm"] = holm(rq5[m].p_delong)
rq5.to_csv(P["stats"] / "rq5_language_adapted_scorer.csv", index=False); rq5.round(4)